# Lines

**Geometry type:** `line`

Independent line segment pairs — synaptic contact sites, vessel endpoints,
pairwise distance annotations.

1. Generate synthetic line segments (synapse-like pairs)
2. Write to a `.zarrvectors` store
3. Open lazily and inspect metadata
4. Read all segments
5. Spatial bounding-box query
6. Validate


In [ ]:
import numpy as np, tempfile, os
from zarr_vectors.lazy import open_zvr
from zarr_vectors.types.lines import write_lines, read_lines
from zarr_vectors.validate import validate

_tmpdir = tempfile.mkdtemp(prefix="zvf_examples_")
STORE = os.path.join(_tmpdir, "synapses.zarrvectors")
print("Store path:", STORE)

## 1 · Generate synthetic contact-site pairs

In [ ]:
rng = np.random.default_rng(0)
N = 5_000  # synapse pairs

# Synapse midpoints distributed across a 500³ µm volume
midpoints = rng.uniform(50, 450, (N, 3)).astype(np.float32)
# Each synapse is a short segment (1–3 µm) between pre- and post-synaptic sites
offsets = rng.uniform(-1.5, 1.5, (N, 3)).astype(np.float32)

pre_pos  = midpoints - offsets   # (N, 3)
post_pos = midpoints + offsets   # (N, 3)

# `write_lines` expects endpoints shaped (N, 2, D):
#   endpoints[i, 0] = pre, endpoints[i, 1] = post
endpoints = np.stack([pre_pos, post_pos], axis=1).astype(np.float32)

# Per-vertex (per-endpoint) attribute: weight, shape (N, 2)
weights = rng.uniform(0, 1, (N, 2)).astype(np.float32)
# Per-line attribute: synapse type, shape (N,)
synapse_type = rng.integers(0, 4, N).astype(np.int32)

print(f"endpoints   : {endpoints.shape}   (N, 2, D)")
print(f"weights     : {weights.shape}   (per-endpoint)")
print(f"synapse_type: {synapse_type.shape}   (per-line)")
seg_lengths = np.linalg.norm(endpoints[:, 1] - endpoints[:, 0], axis=1)
print(f"Segment length range: {seg_lengths.min():.2f} – {seg_lengths.max():.2f} µm")

## 2 · Write to ZVF store

In [ ]:
write_lines(
    STORE,
    endpoints,
    chunk_shape=(100.0, 100.0, 100.0),
    bin_shape=(25.0, 25.0, 25.0),
    attributes={"weight": weights},          # per-endpoint, (N, 2)
    line_attributes={"synapse_type": synapse_type},   # per-line, (N,)
)
print("Write complete.")

## 3 · Open lazily and inspect metadata

In [ ]:
store = open_zvr(STORE)

print(f"geometry_types: {store.geometry_types}")
print(f"ndim          : {store.ndim}")
print(f"chunk_shape   : {store.chunk_shape}")
print(f"vertex count  : {store[0].vertex_count:,}  (= {N} lines × 2 endpoints)")

## 4 · Read all segments

In [ ]:
result = read_lines(STORE)
print(f"line_count : {result['line_count']:,}")
print(f"endpoints  : {result['endpoints'].shape}   (M, 2, D)")

# Recover (pre, post) pairs directly from endpoints
pre  = result["endpoints"][:, 0]
post = result["endpoints"][:, 1]
lengths = np.linalg.norm(post - pre, axis=1)
print(f"\nSegment length: mean={lengths.mean():.2f} µm, std={lengths.std():.2f} µm")

## 5 · Spatial bounding-box query

Lines are returned if any endpoint falls inside the bounding box.

In [ ]:
lo = np.array([200.0, 200.0, 200.0])
hi = np.array([300.0, 300.0, 300.0])

bbox_result = read_lines(STORE, bbox=(lo, hi))
print(f"Lines with an endpoint in 100³ µm bbox: {bbox_result['line_count']:,}")

# Verify: at least one endpoint of each returned line lies inside the bbox
ep = bbox_result["endpoints"]                      # (M, 2, 3)
in_box = ((ep >= lo) & (ep <= hi)).all(axis=2)    # (M, 2) bool
either_in = in_box.any(axis=1)                     # (M,)
print(f"Lines with ≥1 endpoint in bbox: {either_in.sum()} / {len(either_in)}")

## 6 · Validate

In [ ]:
result_v = validate(STORE, level=3)
print(result_v.summary())


## Summary

`line` stores independent finite line segments. Each line is a pair of
endpoints; both endpoints must lie in the same chunk for direct write.
Use `polyline` for paths that span multiple chunks.

| Step | API |
|------|-----|
| Write | `write_lines(path, endpoints, chunk_shape, bin_shape, attributes, line_attributes)` |
| Read all | `read_lines(path)` → `{"endpoints": (M,2,D), "line_count": M}` |
| Bbox query | `read_lines(path, bbox=(lo, hi))` |
| Read by ID | `read_lines(path, object_ids=[...])` |